# RAID Leaderboard Submission — Fast-DetectGPT (free GPU)

**What this does**
1. Installs the RAID package + models (free).
2. **Bake-off:** scores a small *labeled* RAID sample with three configs and picks the best by AUROC:
   - `gpt2` (124M, single model) — what the live site uses
   - `gpt2-medium` (355M, single model) — bigger, better proxy for modern generators
   - **two-model** `gpt-neo-125M` (reference) + `gpt2` (scoring) — the paper's stronger setup
3. Runs the **winning** config over the full RAID test set → `predictions.json`.
4. Writes `metadata.json` and zips both for the pull request.

**Before you run:** Runtime ▸ Change runtime type ▸ **GPU (T4)**. Then Runtime ▸ Run all.
Total time ≈ 1 hour, cost = $0.

In [ ]:
# 1) Install + confirm GPU
!pip -q install raid-bench transformers torch scikit-learn tqdm
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU (T4), then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2) Batched Fast-DetectGPT. One function handles single-model AND two-model.
#    Same math as backend/fast_detect.py, vectorized over a batch and with an
#    optional separate reference model (shared GPT-2 tokenizer across all three).
import torch, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from functools import lru_cache

DEVICE = 'cuda'
MAX_TOKENS = 1024
BATCH = 16

@lru_cache(maxsize=4)
def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(name).to(DEVICE).eval().half()
    return tok, m

@torch.no_grad()
def score_batch(texts, scoring='gpt2', reference=None):
    """Return a list of discrepancy floats (higher = more machine-like).
    reference=None -> single-model variant (reference == scoring)."""
    tok, m_score = load(scoring)
    _, m_ref = load(reference) if reference else (tok, m_score)
    out = []
    for i in range(0, len(texts), BATCH):
        chunk = [t if t and t.strip() else ' ' for t in texts[i:i+BATCH]]
        enc = tok(chunk, return_tensors='pt', truncation=True, max_length=MAX_TOKENS,
                  padding=True).to(DEVICE)
        ids, attn = enc.input_ids, enc.attention_mask
        lg_s = m_score(ids, attention_mask=attn).logits[:, :-1, :].float()
        lg_r = lg_s if reference is None else m_ref(ids, attention_mask=attn).logits[:, :-1, :].float()
        tgt = ids[:, 1:]
        mask = attn[:, 1:].bool()
        logp_s = torch.log_softmax(lg_s, dim=-1)            # scoring log-probs
        q = torch.softmax(lg_r, dim=-1)                     # reference dist
        lp = logp_s.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)          # actual-token logprob
        mu = (q * logp_s).sum(-1)                                       # E_q[logp_s]
        e2 = (q * logp_s.pow(2)).sum(-1)
        var = (e2 - mu.pow(2)).clamp(min=1e-8)
        for b in range(ids.shape[0]):
            mb = mask[b]
            if mb.sum() < 4: out.append(0.0); continue
            num = (lp[b][mb].sum() - mu[b][mb].sum())
            den = var[b][mb].sum().sqrt()
            out.append((num/den).item() if den > 0 else 0.0)
    return out

CONFIGS = {
    'gpt2_single':      dict(scoring='gpt2'),
    'gpt2medium_single':dict(scoring='gpt2-medium'),
    'twomodel_neo_gpt2':dict(scoring='gpt2', reference='EleutherAI/gpt-neo-125m'),
}
print('configs:', list(CONFIGS))

In [ ]:
# 3) BAKE-OFF on a small labeled RAID sample -> pick the config with best AUROC.
import numpy as np, random
from raid.utils import load_data
from sklearn.metrics import roc_auc_score

random.seed(0)
train = load_data(split='train')
# Balanced ~300-row sample across generators; label 1 = machine, 0 = human.
rows = train.to_dict('records') if hasattr(train, 'to_dict') else list(train)
def is_ai(r): return 0 if str(r.get('model','')).lower() in ('human','') else 1
ai  = [r for r in rows if is_ai(r)==1]
hum = [r for r in rows if is_ai(r)==0]
random.shuffle(ai); random.shuffle(hum)
sample = ai[:150] + hum[:150]
random.shuffle(sample)
texts  = [r['generation'] if 'generation' in r else r.get('text','') for r in sample]
labels = [is_ai(r) for r in sample]
print(f'bake-off sample: {sum(labels)} AI / {len(labels)-sum(labels)} human')

results = {}
for name, cfg in CONFIGS.items():
    s = score_batch(texts, **cfg)
    auc = roc_auc_score(labels, s)
    results[name] = auc
    print(f'  {name:20s} AUROC {auc:.4f}')
winner = max(results, key=results.get)
print(f'\nWINNER: {winner}  (AUROC {results[winner]:.4f})')

In [ ]:
# 4) Full RAID TEST run with the winning config -> predictions.json
#    (This is the slow cell — ~1hr on T4. It scores the whole hidden test set.)
import json
from raid import run_detection
from raid.utils import load_data

cfg = CONFIGS[winner]
def my_detector(texts):
    return score_batch(list(texts), **cfg)

test_df = load_data(split='test')
predictions = run_detection(my_detector, test_df)
with open('predictions.json','w') as f: json.dump(predictions, f)
print('wrote predictions.json for config:', winner)

In [ ]:
# 5) metadata.json + zip for the pull request
import json, zipfile
meta = {
  'name': 'fast-detectgpt-lite',
  'detector': 'Fast-DetectGPT (conditional probability curvature, Bao et al. 2024)',
  'configuration': winner,
  'bakeoff_auroc': {k: round(v,4) for k,v in results.items()},
  'description': ('Single-/two-model Fast-DetectGPT on small open models (GPT-2 / '
                  'GPT-Neo-125M), selected by a labeled bake-off. Zero-cost, free-GPU '
                  'reproducible. Part of a layered detection tool: '
                  'https://github.com/Matman22/Ai-Layered-Detection-Tool'),
  'features': 'zero-shot; no training on RAID',
  'organization': 'independent',
  'open_source': True,
  'url': 'https://github.com/Matman22/Ai-Layered-Detection-Tool'
}
with open('metadata.json','w') as f: json.dump(meta, f, indent=2)
with zipfile.ZipFile('raid_submission.zip','w') as z:
    z.write('predictions.json'); z.write('metadata.json')
print(json.dumps(meta, indent=2))
try:
    from google.colab import files; files.download('raid_submission.zip')
except Exception as e:
    print('download manually from the file browser:', e)